# Matrix Computation: NumPy and SymPy

This notebook accompanies the **Compute responsibly** section of Lecture 00. It is a first orientation to matrix computation—not a Python reference manual. Run the cells in order, change small examples, and return here when you need a familiar pattern later in the course.

We will use two complementary packages:

- **NumPy** for efficient numerical computation with floating-point numbers and realistic problems.
- **SymPy** for exact arithmetic, symbolic computation, algebraic verification, and textbook-like displays.

It is common to compute numerically with NumPy and display a small result with SymPy's `Matrix(...)`. That conversion improves the presentation; it does **not** restore precision that floating-point computation has already lost.

In [ ]:
import numpy as np
import sympy as sp
from sympy import Matrix
import matplotlib.pyplot as plt
from IPython.display import Math
import classlib as cl

%matplotlib inline

cl.nbviz.init(use_tex=True)
colors = cl.nbviz.TOL_BRIGHT

### Display helpers

These two small helpers use SymPy only for textbook-style display. They accept NumPy arrays or SymPy matrices and do not change the underlying calculation.

In [ ]:
def show_augmented(A, b):
    """Return the augmented matrix [A | b] with a visible divider."""
    A_display = Matrix(A)
    b_display = Matrix(b)
    if A_display.rows != b_display.rows:
        raise ValueError("A and b must have the same number of rows")
    entries = [
        list(A_display.row(i)) + list(b_display.row(i))
        for i in range(A_display.rows)
    ]
    body = r" \\ ".join(
        " & ".join(sp.latex(entry) for entry in row) for row in entries
    )
    columns = "c" * A_display.cols + "|" + "c" * b_display.cols
    return Math(rf"\left[\begin{{array}}{{{columns}}}{body}\end{{array}}\right]")


def show_blocks(*block_rows):
    """Return a symbolic block matrix; pass one list of blocks per row."""
    return sp.BlockMatrix([[Matrix(block) for block in row] for row in block_rows])

## 1. Creating vectors and matrices

A vector in $\mathbb{R}^n$ records $n$ numbers. A matrix in $\mathbb{R}^{m\times n}$ is a rectangular array with $m$ rows and $n$ columns. NumPy's `array` is the basic numerical container.

In [ ]:
x = np.array([3.0, 2.0])                 # one-dimensional array
row = np.array([[3.0, 2.0]])             # 1 x 2 row vector
column = np.array([[3.0], [2.0]])        # 2 x 1 column vector
A = np.array([[1.0, 0.5], [0.0, 1.0]])

Matrix(x), Matrix(row), Matrix(column), Matrix(A)

A NumPy vector written as `np.array([3, 2])` has shape `(2,)`: it is neither explicitly a row nor explicitly a column. Use a two-dimensional array when that distinction matters.

In [ ]:
print("x.shape     =", x.shape)
print("row.shape   =", row.shape)
print("column.shape=", column.shape)
print("A.shape     =", A.shape)

In [ ]:
Z = np.zeros((2, 3))
O = np.ones((2, 3))
I = np.eye(3)
Matrix(Z), Matrix(O), Matrix(I)

SymPy matrices are always two-dimensional and are designed for exact or symbolic work. Integer entries remain exact.

In [ ]:
x_exact = Matrix([3, 2])
row_exact = Matrix([[3, 2]])
A_exact = Matrix([[1, sp.Rational(1, 2)], [0, 1]])
Z_exact = sp.zeros(2, 3)
O_exact = sp.ones(2, 3)
I_exact = sp.eye(3)

print("A_exact.shape =", A_exact.shape)
A_exact, Z_exact, O_exact, I_exact

## 2. Accessing entries

Python starts counting at zero. Thus `A[0, 1]` is the entry in the first row and second column. Slices use `start:stop`, including `start` but excluding `stop`.

In [ ]:
A = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0],
              [7.0, 8.0, 9.0]])

print("entry A[0, 1]:", A[0, 1])
print("first row:    ", A[0, :])
print("second column:", A[:, 1])
Matrix(A[:2, 1:])

In [ ]:
A_changed = A.copy()
A_changed[0, 1] = -2
A_changed[:, 2] = 0
Matrix(A_changed)

SymPy uses similar indexing. Use two-dimensional slices to retain a row or column as a matrix.

In [ ]:
S = Matrix([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
S_changed = S.copy()
S_changed[0, 1] = -2
S[0, :], S[:, 1], S_changed

## 3. Matrix arithmetic

For compatible matrices, addition and scalar multiplication work as expected. The crucial NumPy distinction is:

- `A * B` multiplies corresponding entries (**elementwise multiplication**).
- `A @ B` performs **matrix multiplication**.

Use `@` for the linear-algebra products $A\boldsymbol{x}$ and $AB$.

In [ ]:
A = np.array([[1.0, 2.0], [3.0, 4.0]])
B = np.array([[2.0, 0.0], [1.0, 2.0]])
x = np.array([1.0, -1.0])

Matrix(A + B), Matrix(3 * A), Matrix(A * B), Matrix(A @ B), Matrix(A @ x)

In [ ]:
Matrix(A.T), Matrix(B @ A), Matrix(A @ B)

The transpose $A^{\mathsf T}$ exchanges rows and columns. Matrix multiplication usually does not commute: here $BA\ne AB$. In SymPy, `*` means matrix multiplication when both operands are matrices.

In [ ]:
A_exact = Matrix([[1, 2], [3, 4]])
B_exact = Matrix([[2, 0], [1, 2]])
x_exact = Matrix([1, -1])
A_exact + B_exact, 3 * A_exact, A_exact * x_exact, A_exact * B_exact, A_exact.T

## 4. Common matrix construction utilities

`reshape` changes the arrangement without changing the entries. Stacking and concatenation assemble larger arrays from compatible pieces.

In [ ]:
numbers = np.arange(1, 7)
R = numbers.reshape(2, 3)
u = np.array([[1.0], [2.0]])
v = np.array([[3.0], [4.0]])

Matrix(R), Matrix(np.hstack((u, v))), Matrix(np.vstack((u.T, v.T)))

In [ ]:
top = np.array([[1.0, 2.0]])
bottom = np.array([[3.0, 4.0]])
D = np.diag([2.0, 5.0, 8.0])

Matrix(np.concatenate((top, bottom), axis=0)), Matrix(D), np.diag(D)

In SymPy, `row_join` places matrices side by side and `col_join` places them one above another.

In [ ]:
u_exact = Matrix([1, 2])
v_exact = Matrix([3, 4])
u_exact.row_join(v_exact), u_exact.T.col_join(v_exact.T), sp.diag(2, 5, 8)

In [ ]:
show_blocks(
    [sp.eye(2), Matrix([1, 2])],
    [Matrix([[3, 4]]), Matrix([[5]])],
)

## 5. Solving linear systems

Consider $A\boldsymbol{x}=\boldsymbol{b}$. NumPy's `np.linalg.solve(A, b)` computes a numerical solution of a square system.

In [ ]:
A = np.array([[3.0, 1.0], [1.0, 2.0]])
b = np.array([9.0, 8.0])
x = np.linalg.solve(A, b)

print("solution x =", x)
show_augmented(A, b)

In [ ]:
A_exact = Matrix([[3, 1], [1, 2]])
b_exact = Matrix([9, 8])
x_exact = A_exact.LUsolve(b_exact)
x_exact

Mathematically, $\boldsymbol{x}=A^{-1}\boldsymbol{b}$ when $A^{-1}$ exists. Computationally, use `solve` or `LUsolve`: explicitly forming the inverse generally costs more and can introduce more numerical error.

In [ ]:
x_from_inverse = np.linalg.inv(A) @ b
print("solve result:  ", x)
print("inverse result:", x_from_inverse)

## 6. Determinant, inverse, and rank

For a square matrix, the **determinant** helps detect whether the associated transformation is reversible. The **rank** counts independent directions of action or independent information. An **inverse** reverses the action of an invertible square matrix.

In [ ]:
A = np.array([[3.0, 1.0], [1.0, 2.0]])
print("determinant:", np.linalg.det(A))
print("rank:       ", np.linalg.matrix_rank(A))
Matrix(np.linalg.inv(A))

In [ ]:
A_exact = Matrix([[3, 1], [1, 2]])
A_exact.det(), A_exact.rank(), A_exact.inv()

## 7. Exact arithmetic versus floating point

Computers store most real numbers using finite-precision approximations. NumPy's floating-point arithmetic is fast and appropriate for realistic numerical work. SymPy can preserve rational numbers exactly, which is valuable for demonstrations and algebraic checks.

In [ ]:
third_float = np.float64(1) / 3
third_exact = sp.Rational(1, 3)

print("NumPy 1/3:       ", format(third_float, ".20f"))
print("NumPy 3*(1/3)-1:", 3 * third_float - 1)
third_exact, 3 * third_exact - 1

Now convert an already approximate NumPy value into a SymPy matrix. The display changes, but the stored approximation does not become the exact rational number $1/3$.

In [ ]:
approximate_matrix = Matrix(np.array([[third_float]]))
exact_matrix = Matrix([[third_exact]])
approximate_matrix, exact_matrix, approximate_matrix[0, 0] == exact_matrix[0, 0]

## 8. Least squares

Lecture 00 represents a fitted line $y\approx c_0+c_1t$ as an overdetermined system $A\boldsymbol{c}\approx\boldsymbol{b}$. Usually no coefficient vector fits every data point exactly. `np.linalg.lstsq` finds the least-squares solution $\widehat{\boldsymbol{c}}$ that minimizes the Euclidean residual norm

$$
\lVert A\boldsymbol{c}-\boldsymbol{b}\rVert_2.
$$

In [ ]:
t = np.array([0.0, 1.0, 2.0, 3.0])
b = np.array([1.0, 2.0, 2.0, 4.0])
A = np.column_stack((np.ones_like(t), t))

c, squared_residuals, rank, singular_values = np.linalg.lstsq(A, b, rcond=None)
fitted = A @ c
residual = b - fitted

print("fitted coefficients [c0, c1]:", c)
print("residual vector:             ", residual)
print("residual norm:               ", np.linalg.norm(residual))
Matrix(A), Matrix(c), Matrix(residual)

In [ ]:
plt.scatter(t, b, color=colors["blue"], label="data")
plt.plot(t, fitted, color=colors["cyan"], label="least-squares line")
plt.xlabel(r"$t$")
plt.ylabel(r"$b$")
plt.legend(frameon=False);

## 9. Compute responsibly

Reliable computation requires more than obtaining an answer:

1. Prefer a solver such as `solve` or `lstsq` to explicitly forming an inverse.
2. Remember that floating-point results are approximations.
3. Mathematically equivalent formulas can behave differently in finite precision.
4. Check matrix shapes before multiplying or stacking arrays.
5. Check a result with a residual such as $\lVert A\boldsymbol{x}-\boldsymbol{b}\rVert_2$.

In [ ]:
A = np.array([[3.0, 1.0], [1.0, 2.0]])
b = np.array([9.0, 8.0])
x = np.linalg.solve(A, b)

print("A.shape:      ", A.shape)
print("x.shape:      ", x.shape)
print("b.shape:      ", b.shape)
print("residual norm:", np.linalg.norm(A @ x - b))

Two algebraically equivalent formulas from Lecture 00 illustrate catastrophic cancellation. The rationalized form avoids subtracting nearly equal floating-point numbers.

In [ ]:
small = 1.23456789 * np.array([1e-6, 1e-7, 1e-8])
direct = np.sqrt(1 + small**2) - 1
rationalized = small**2 / (np.sqrt(1 + small**2) + 1)

np.column_stack((small, direct, rationalized))

The condition number previews sensitivity: a large value warns that small changes in the data may produce much larger changes in the solution. Conditioning is a property of the mathematical problem; reliable software cannot remove it.

In [ ]:
well_conditioned = np.array([[3.0, 1.0], [1.0, 2.0]])
nearly_singular = np.array([[1.0, 1.0], [1.0, 1.000001]])

print("condition number, first matrix: ", np.linalg.cond(well_conditioned))
print("condition number, nearly singular:", np.linalg.cond(nearly_singular))

### A habit to keep

Before trusting a computed answer, ask: Is the mathematical problem well posed? Is it sensitive to its data? Is the algorithm appropriate and efficient? Do the shapes match? Is the residual small?

Throughout MATH 332, use computation to explore and verify linear algebra—not as a substitute for understanding why a method works.